# Clase 2. Visión por computador I: redes convolucionales

**Advanced Deep Learning**

*Siglas usadas en esta clase: red neuronal convolucional (CNN), unidad lineal rectificada (ReLU), unidad de procesamiento gráfico (GPU), rojo-verde-azul (RGB), descenso de gradiente estocástico (SGD).*

## Objetivos de la clase

Al terminar esta clase, sabrás:

- Explicar por qué una red totalmente conectada no funciona bien con imágenes
- Describir cómo actúa un filtro convolucional sobre una imagen, paso a paso
- Calcular el tamaño de salida de una capa convolucional a partir del *kernel*, el *stride* y el *padding*
- Explicar para qué sirve el *pooling* dentro de una CNN
- Construir y entrenar una red neuronal convolucional (CNN) pequeña con PyTorch sobre FashionMNIST
- Visualizar qué ha aprendido la primera capa de una CNN ya entrenada

## Por qué una red normal no basta para imágenes

- Para dar una imagen a una red totalmente conectada, primero la aplanas (*flatten*): conviertes la malla de píxeles en un único vector
- Al aplanar, pierdes la estructura espacial: dos píxeles vecinos en la imagen original pueden acabar muy separados en el vector
- El número de parámetros se dispara. Una imagen de 32 x 32 píxeles en color (RGB) tiene 3.072 valores de entrada; conectados a 1.000 neuronas de la primera capa oculta ya suman 3.072.000 parámetros
- Las imágenes naturales cumplen dos propiedades que una CNN aprovecha:
  - **Localidad**: los píxeles cercanos suelen estar correlacionados (forman parte del mismo objeto o textura)
  - **Invariancia a la traslación**: un patrón (un ojo, un borde) puede aparecer en cualquier posición de la imagen
- Una capa convolucional explota ambas propiedades:
  - Conecta cada unidad de salida solo con una región local de la entrada, llamada **campo receptivo**
  - Reutiliza los mismos pesos en toda la imagen: **compartición de pesos** (*weight sharing*)

## El filtro convolucional en acción

- Un filtro (*kernel*) es una pequeña matriz de pesos, por ejemplo de 3 x 3
- Se desliza sobre la imagen: en cada posición multiplica sus valores por los píxeles que tiene debajo y suma el resultado
- Si la región se parece al patrón del filtro, la suma da un valor alto; si no se parece, da un valor bajo
- El resultado de recorrer toda la imagen es un **mapa de activaciones** (*feature map*)
- En una CNN entrenada, los valores del filtro no los diseña una persona: los aprende la red, igual que los pesos de una capa lineal

Abajo aplicas a mano, con NumPy, un detector de bordes verticales sobre una imagen sintética pequeña. Fíjate en el tamaño de salida: con una entrada de 9 x 9 y un filtro de 3 x 3 sin relleno, obtienes 7 x 7 (convolución "válida").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Imagen sintética de 9x9: mitad izquierda oscura, mitad derecha clara -> un borde vertical
imagen = np.zeros((9, 9), dtype=np.float32)
imagen[:, 5:] = 1.0

# Filtro detector de bordes verticales (tipo Sobel)
kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

def convolucion_manual(imagen, kernel):
    f = kernel.shape[0]
    n = imagen.shape[0]
    salida = np.zeros((n - f + 1, n - f + 1), dtype=np.float32)
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            region = imagen[i:i + f, j:j + f]
            salida[i, j] = np.sum(region * kernel)
    return salida

mapa_activaciones = convolucion_manual(imagen, kernel)

fig, ejes = plt.subplots(1, 2, figsize=(8, 4))
ejes[0].imshow(imagen, cmap="gray")
ejes[0].set_title("Imagen original (borde vertical)")
ejes[1].imshow(mapa_activaciones, cmap="gray")
ejes[1].set_title("Mapa de activaciones")
for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

print(f"Tamaño de entrada: {imagen.shape} | Tamaño de salida: {mapa_activaciones.shape}")

## Stride, padding y pooling

- **Convolución válida (*valid*)**: sin relleno, la salida siempre es más pequeña que la entrada: $O = N - F + 1$
- **Padding (relleno)**: añades ceros alrededor del borde de la imagen; con el relleno justo obtienes una convolución "igual" (*same*), que mantiene el tamaño de entrada y salida
- **Stride (paso)**: controla cada cuántos píxeles se desplaza el filtro; con *stride* dos, calculas solo la mitad de posiciones en cada dimensión
- Fórmula general, combinando *kernel* ($F$), *padding* ($P$) y *stride* ($S$): $O = \lfloor (N - F + 2P) / S \rfloor + 1$
- **Pooling** (típicamente *max pooling* de 2 x 2) reduce la resolución espacial de los mapas de activaciones sin añadir parámetros que aprender

In [ ]:
import torch
import torch.nn as nn

entrada = torch.randn(1, 1, 9, 9)  # (lote, canales, alto, ancho)
print(f"Tamaño de entrada: {tuple(entrada.shape)}\n")

configuraciones = [
    {"kernel_size": 3, "stride": 1, "padding": 0},  # convolución válida
    {"kernel_size": 3, "stride": 1, "padding": 1},  # convolución "igual" (same)
    {"kernel_size": 3, "stride": 2, "padding": 0},  # stride dos
    {"kernel_size": 5, "stride": 1, "padding": 2},  # kernel más grande, "igual"
]

for config in configuraciones:
    capa = nn.Conv2d(in_channels=1, out_channels=1, **config)
    salida = capa(entrada)
    print(f"kernel={config['kernel_size']} stride={config['stride']} "
          f"padding={config['padding']} -> salida {tuple(salida.shape)}")

# Pooling: reduce alto y ancho a la mitad, sin parámetros que aprender
pool = nn.MaxPool2d(kernel_size=2, stride=2)
mapa_ejemplo = torch.randn(1, 32, 32, 32)
print(f"\nAntes de max pooling: {tuple(mapa_ejemplo.shape)}")
print(f"Después de max pooling 2x2: {tuple(pool(mapa_ejemplo).shape)}")

## Construye tu CNN

- Una CNN apila bloques de convolución + ReLU + *pooling*, y termina en una o varias capas totalmente conectadas
- Entrenas sobre FashionMNIST: imágenes en escala de grises de 28 x 28 píxeles con diez categorías de prendas de ropa, disponible directamente en `torchvision`
- El entrenamiento sigue el mismo procedimiento que ya conoces: pérdida (entropía cruzada), retropropagación y actualización de pesos con un optimizador (aquí, Adam, una variante de SGD)
- Si Colab te asigna una GPU, el entrenamiento va más rápido; si no, funciona igualmente sobre CPU

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

clases = [
    "camiseta", "pantalón", "jersey", "vestido", "abrigo",
    "sandalia", "camisa", "zapatilla", "bolso", "botín",
]

In [ ]:
class CNNSimple(nn.Module):
    def __init__(self, num_clases=10):
        super().__init__()
        # Bloque 1: 1 canal de entrada (escala de grises) -> 16 filtros, padding "same"
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16,
                                kernel_size=3, stride=1, padding=1)
        # Bloque 2: 16 -> 32 filtros
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32,
                                kernel_size=3, stride=1, padding=1)
        # Max pooling 2x2: reduce el alto y el ancho a la mitad cada vez
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dos pooling de 2x2 sobre una entrada de 28x28: 28 -> 14 -> 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_clases)
        self.dropout = nn.Dropout(0.25)  # mitiga el sobreajuste

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # -> (lote, 16, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))  # -> (lote, 32, 7, 7)
        x = torch.flatten(x, start_dim=1)     # aplana todo menos el eje de lote
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                       # logits, sin softmax explícito
        return x

modelo = CNNSimple(num_clases=10).to(device)
print(modelo)

In [ ]:
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        optimizador.zero_grad()
        salidas = modelo(imagenes)
        perdida = criterio(salidas, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)

def evaluar(modelo, loader, criterio):
    modelo.eval()
    perdida_total = 0.0
    aciertos = 0
    with torch.no_grad():
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            perdida_total += criterio(salidas, etiquetas).item() * imagenes.size(0)
            predicciones = salidas.argmax(dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
    return perdida_total / len(loader.dataset), aciertos / len(loader.dataset)

# Pocas épocas: en clase priorizamos ver el proceso completo funcionando, no exprimir la exactitud
num_epocas = 3
perdidas_train = []
perdidas_test = []

for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    perdida_test, exactitud_test = evaluar(modelo, test_loader, criterio)
    perdidas_train.append(perdida_train)
    perdidas_test.append(perdida_test)
    print(f"Época {epoca}/{num_epocas} | "
          f"pérdida train: {perdida_train:.4f} | "
          f"pérdida test: {perdida_test:.4f} | "
          f"exactitud test: {exactitud_test:.2%}")

print(f"\nExactitud final sobre el conjunto de prueba: {exactitud_test:.2%}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epocas + 1), perdidas_train, marker="o", label="Pérdida train")
plt.plot(range(1, num_epocas + 1), perdidas_test, marker="o", label="Pérdida test")
plt.xlabel("Época")
plt.ylabel("Pérdida (entropía cruzada)")
plt.title("Curva de pérdida durante el entrenamiento")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Qué ha aprendido la red

- La primera capa convolucional aprende, casi siempre, detectores simples: bordes y cambios de intensidad
- Puedes visualizar directamente los pesos de esos filtros, porque cada uno es una pequeña matriz, igual que el `kernel` que aplicaste a mano al principio de la clase

In [ ]:
pesos_conv1 = modelo.conv1.weight.data.cpu().clone()  # forma (16, 1, 3, 3)
num_filtros = pesos_conv1.shape[0]

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.5))
for indice, eje in enumerate(ejes.flat):
    filtro = pesos_conv1[indice, 0]
    eje.imshow(filtro, cmap="gray")
    eje.set_title(f"Filtro {indice}", fontsize=8)
    eje.axis("off")

plt.suptitle("Filtros aprendidos en la primera capa convolucional")
plt.tight_layout()
plt.show()

## Resumen de la clase

- Una red totalmente conectada no escala a imágenes: destruye la estructura espacial al aplanar la entrada y necesita demasiados parámetros
- Un filtro convolucional es una plantilla de pesos aprendibles que recorre la imagen buscando un patrón
- La compartición de pesos reduce drásticamente el número de parámetros y hace la red equivariante a la traslación
- El *stride*, el *padding* y el *pooling* controlan el tamaño espacial de los mapas de activaciones a lo largo de la red
- Una CNN se construye apilando bloques de convolución, ReLU y *pooling*, y termina en capas totalmente conectadas
- Al apilar capas convolucionales, el campo receptivo crece: primero bordes y texturas, luego formas, después objetos completos

**Abre ahora el cuaderno de práctica `../notebooks_practica/clase02_practica_cnn_vision_por_computador.ipynb` y ponlo en marcha.**